<h1 style=\"text-align: center; font-size: 50px;\">🌷 Register Model with LDA and SVM </h1>
This notebook is about Iris Flowers: a famous machine learning classification problem. <br>
The goal is to create a model that classifies the categorical variable (setosa, virginica or versicolor) based in some probability.

## Notebook Overview
- Imports
- Configurations
- Define User Constants
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference


## Imports

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

# ------------------------ Data Manipulation ------------------------
import numpy as np
import pandas as pd

# ------------------------ System Utilities ------------------------
import os
import warnings
import logging
import time
from typing import Optional, Any
import sys

# ------------------------ Machine Learning tools ------------------------
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

# ------------------------ MLflow for Experiment Tracking and Model Management ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Define the relative path to the 'src' directory (one level up from current working directory)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.mlflow import Logger

from src.utils import (
    load_config,
)



Note: you may need to restart the kernel to use updated packages.
CPU times: user 2.64 s, sys: 5.81 s, total: 8.45 s
Wall time: 2.79 s


## Configurations

In [2]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [3]:
# Create logger
logger = logging.getLogger("flower_logger")
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", 
                              datefmt="%Y-%m-%d %H:%M:%S")  

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

## Define User Constants

In [4]:
# ------------------------- Paths -------------------------
DATASET_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv"

ARTIFACT_PATH = "iris_model"
CONFIG_PATH = "../configs/config.yaml"
DEMO_FOLDER = "../demo"
# ------------------------ MLflow Integration ------------------------
EXPERIMENT_NAME = "Iris_Flower_Experiment"
RUN_NAME = "Iris_Flower_Run"
MODEL_NAME = "Iris_Flower_Model"

In [5]:
# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")


✅ Configuration loaded successfully


In [6]:
start_time = time.time() 
logger.info('Notebook execution started.')

2026-04-14 21:46:25 - INFO - Notebook execution started.


## Logging Model to MLflow

In [7]:

%%time

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", '/phoenix/mlflow'))
# === Set MLflow experiment context ===
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

logger.info(f'Starting the experiment: {EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Start an MLflow run
input_schema = Schema([
    ColSpec("double","sepal-length"),
    ColSpec("double","sepal-width"),
    ColSpec("double","petal-length"),
    ColSpec("double","petal-width"),
    ])
output_schema = Schema([
    ColSpec("string", "class"),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name=RUN_NAME) as run:
    logger.info(f"Starting the experiment: {EXPERIMENT_NAME}")
    
    # Log the model using our custom class
    Logger.log_model(
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        demo_folder=DEMO_FOLDER,
        signature=signature,

    )
    
    # Register the model in the MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(
        model_uri=model_uri, 
        name=MODEL_NAME
    )

logger.info(f'Registered the model: {MODEL_NAME}')

2026-04-14 21:46:25 - INFO - Starting the experiment: Iris_Flower_Experiment
2026-04-14 21:46:25 - INFO - Using MLflow tracking URI: /phoenix/mlflow
2026-04-14 21:46:25 - INFO - Starting the experiment: Iris_Flower_Experiment
Registered model 'Iris_Flower_Model' already exists. Creating a new version of this model...
2026/04/14 21:46:28 WARNING mlflow.tracking._model_registry.fluent: Run with id 57974da0120142898264bfdac59abdd5 has no artifacts at artifact path 'Iris_Flower_Model', registering model based on models:/m-d237ce999b0e446cb9b949d064602969 instead
Created version '4' of model 'Iris_Flower_Model'.
2026-04-14 21:46:28 - INFO - Registered the model: Iris_Flower_Model


CPU times: user 636 ms, sys: 253 ms, total: 889 ms
Wall time: 3.6 s


## Fetching the Latest Model Version from MLflow

In [8]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the "Iris_Flower_Model" model (not yet in a specific stage)
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
logger.info(f"Latest Model Version: {latest_model_version}")
logger.info(f"Model Signature: {model_info.signature}")

2026-04-14 21:46:29 - INFO - Latest Model Version: 4
2026-04-14 21:46:29 - INFO - Model Signature: inputs: 
  ['sepal-length': double (required), 'sepal-width': double (required), 'petal-length': double (required), 'petal-width': double (required)]
outputs: 
  ['class': string (required)]
params: 
  None



## Loading the Model and Running Inference

In [9]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")

df_input = pd.DataFrame({
    'sepal-length': [5.1],
    'sepal-width': [3.5],
    'petal-length':	[1.4],
    'petal-width': [0.2]
})
prediction = model.predict(df_input)
logger.info(prediction)


2026-04-14 21:46:30 - INFO - ['Iris-setosa']


In [10]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60
logger.info(f"Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-14 21:46:30 - INFO - Total execution time: 0m 5.15s


In [11]:
print("Notebook execution completed successfully.")

Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).